# Data Cleaning
This notebook is for loading and cleaning of datasets for model building and testing.

**NOTE: `AWC_stream_binary` describes fighter (=`1`) vs non-fighter (=`0`)**

In [1]:
import pandas as pd
import re

## Constants

In [2]:
MODULE_1 = [
    "GH 1",
    "GH 2",
    "GH 3",
    "GH 4",
    "GH 5",
    "GH 6",
    "GH 7",
    "GH 8",
    "GH 9",
    "GH 10",
    "GH 11",
    "GH 12",
    "GH 13",
    "GH 14",
    "GH 15",
    "GH 17",
    "GH 19",
    "GH 21",
    "GH 22",
]

MODULE_2 = [
    "GH 24",
    "GH 25",
    "GH 27",
    "GH 28",
    "GH 29",
    "GH 30",
    "GH 31",
    "GH 32",
    "GH 33",
    "GH 34",
    "GHT",
]

MODULE_3 = ["IF 1", "IF 2", "IF 3", "IF 4", "IF 5", "IF 6", "IFT"]

MODULE_6 = ["AN 1", "AN 2", "AN 3", "AN 4", "AN 5", "AN 6", "AN 7"]

LESSONS_TO_KEEP = MODULE_1 + MODULE_2 + MODULE_3

stream_unit_dict = {
    "FWC" : 1,
    "RWC": 2,
    "TWC": 3,
    "NFTC": 4,
    "ITAF": 5,
    "IERW": 6,
    "SUPT": 7
}

BWC_status_dict = {
    "Fail": 0,
    "Pass": 1,
    "DNF(SUPT)": 1,
}

AWC_stream_binary_dict = {
    1: 1,
    2: 0,
    3: 0,
    4: 1,
    5: 1,
    6: 0,
    7: 1,
}

## Cleaned data for BWC 160 to 200 (source: Melody) (streaming into Fighters)

In [3]:
df_160t200_clean = pd.read_csv("../data/BWC_160-200.csv").drop("av_creoc", axis=1)
df_160t200_clean

,STUDENT_ID,GH 1,GH 2,GH 3,GH 4,GH 5,GH 6,GH 7,GH 8,GH 9,...,GH 35,IF 1,IF 2,IF 3,IF 4,IF 5,IF 6,IFT,AWC_stream,AWC_stream_binary
0,160CHOOC,4.243429,4.379660,4.674136,4.110342,3.962646,3.928343,4.000000,4.107553,4.304875,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,1
1,160HOS,4.556401,4.476572,4.458939,3.850272,4.074621,4.110372,4.385034,4.229443,4.486146,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,1
2,160HUANGZ,4.924944,4.790186,3.849019,4.290526,4.334886,3.964229,4.805187,4.202279,4.306423,...,4.215157,4.000000,3.877501,4.564103,4.569475,4.678403,4.225446,5.359771,1.0,1
3,160LEEB,4.375837,4.361286,4.589199,4.466058,4.556848,4.470913,4.410200,4.000000,4.509552,...,5.005333,4.466849,4.269385,3.714507,3.930278,3.792458,4.494110,4.813372,1.0,1
4,160ONGY,4.215324,4.123000,4.047606,3.925508,NaN,4.101341,4.365242,4.269397,4.000000,...,4.206454,3.869381,4.456073,3.958280,4.324003,3.819823,3.342057,4.326057,2.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
353,200LENP,NaN,4.435512,5.348143,5.219967,NaN,4.850193,NaN,NaN,4.484864,...,4.248393,4.850349,4.476344,4.617854,4.792710,3.946995,4.430193,4.314005,2.0,0
354,200TEWJ,NaN,NaN,4.921511,NaN,NaN,5.086124,4.853173,NaN,NaN,...,NaN,5.552131,5.111225,4.810760,4.898658,4.164966,4.266431,4.406520,5.0,1
355,200VVARUN,5.714286,5.591837,5.101195,4.696520,4.860699,4.445996,NaN,4.173890,NaN,...,NaN,4.850349,4.607360,4.696551,4.388513,4.277323,4.166171,4.347469,3.0,0
356,200WEEW,NaN,5.153390,4.398210,NaN,NaN,5.120336,NaN,4.671114,NaN,...,3.962937,4.672861,4.656508,4.441327,NaN,3.945963,3.923590,3.742980,3.0,0


In [4]:
awc_keep_columns = df_160t200_clean.columns
awc_keep_columns

Index(['STUDENT_ID', 'GH 1', 'GH 2', 'GH 3', 'GH 4', 'GH 5', 'GH 6', 'GH 7',
       'GH 8', 'GH 9', 'GH 10', 'GH 11', 'GH 12', 'GH 13', 'GH 14', 'GH 15',
       'GH 17', 'GH 19', 'GH 21', 'GH 22', 'GH 24', 'GH 25', 'GH 27', 'GH 28',
       'GH 29', 'GH 30', 'GH 31', 'GH 32', 'GH 33', 'GH 34', 'GH 35', 'IF 1',
       'IF 2', 'IF 3', 'IF 4', 'IF 5', 'IF 6', 'IFT', 'AWC_stream',
       'AWC_stream_binary'],
      dtype='str')

## Early Streaming Data for BWC 160 to 200 (source: Joshua) (Pass/Fail)

In [5]:
df_early_streaming_t201 = pd.read_excel("../data/EarlyStreamingData_200BWC.xlsx")
df_early_streaming_t201

,S/No,name,GH 1,GH 2,GH 3,GH 4,GH 5,GH 6,GH 7,GH 8,...,av_bwcf_TF_hours,av_bwcf_RT_sorties,av_bwcf_RT_hours,av_bwcf_BFM_sorties,av_bwcf_BFM_hours,av_bwcf_CGH,av_bwcf_TF,av_bwcf_RT,av_bwcf_BFM,av_bwcf_TI
0,71.0,160CHANC,4.215324,3.937536,4.000000,3.736023,3.726859,3.747200,3.210893,2.673016,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,72.0,160CHANG,3.926446,3.746286,4.094832,4.036967,4.111887,3.783657,2.528977,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,452.0,160CHOOC,4.243429,4.379660,4.674136,4.110342,3.962646,3.928343,4.000000,3.901540,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,453.0,160HOS,4.572844,4.476572,4.458939,3.850272,4.074621,4.110372,4.385034,4.229443,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,73.0,160HOWW,4.936332,4.061821,3.903645,3.581086,3.133589,3.390447,1.487395,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581,NaN,200TERJ,4.858383,5.098765,4.722091,4.293651,4.182033,3.743067,4.467533,3.699077,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
582,NaN,200TEWJ,5.813411,5.653140,5.094496,5.210907,4.576538,5.086124,4.853173,4.721885,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
583,NaN,200VVARUN,5.714286,5.591837,5.101195,4.696520,4.860699,4.445996,4.272728,4.173890,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
584,NaN,200WEEW,5.054227,5.153390,4.398210,4.774973,4.965671,5.120336,4.776854,4.671114,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Rename columns for consistency
df_early_streaming_t201 = df_early_streaming_t201.rename(columns={"name": "STUDENT_ID",
                                                                  "CREOC": "BWC_status_binary"})

In [7]:
# Create AWC_stream_binary column based on AWC_stream
df_early_streaming_t201["AWC_stream_binary"] = df_early_streaming_t201["AWC_stream"].map(AWC_stream_binary_dict)
df_early_streaming_t201

/var/folders/bk/h4fkjwr93xs0lvghl5fm63l40000gn/T/ipykernel_26616/1471886251.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_early_streaming_t201["AWC_stream_binary"] = df_early_streaming_t201["AWC_stream"].map(AWC_stream_binary_dict)


,S/No,STUDENT_ID,GH 1,GH 2,GH 3,GH 4,GH 5,GH 6,GH 7,GH 8,...,av_bwcf_RT_sorties,av_bwcf_RT_hours,av_bwcf_BFM_sorties,av_bwcf_BFM_hours,av_bwcf_CGH,av_bwcf_TF,av_bwcf_RT,av_bwcf_BFM,av_bwcf_TI,AWC_stream_binary
0,71.0,160CHANC,4.215324,3.937536,4.000000,3.736023,3.726859,3.747200,3.210893,2.673016,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,72.0,160CHANG,3.926446,3.746286,4.094832,4.036967,4.111887,3.783657,2.528977,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,452.0,160CHOOC,4.243429,4.379660,4.674136,4.110342,3.962646,3.928343,4.000000,3.901540,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
3,453.0,160HOS,4.572844,4.476572,4.458939,3.850272,4.074621,4.110372,4.385034,4.229443,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
4,73.0,160HOWW,4.936332,4.061821,3.903645,3.581086,3.133589,3.390447,1.487395,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581,NaN,200TERJ,4.858383,5.098765,4.722091,4.293651,4.182033,3.743067,4.467533,3.699077,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
582,NaN,200TEWJ,5.813411,5.653140,5.094496,5.210907,4.576538,5.086124,4.853173,4.721885,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
583,NaN,200VVARUN,5.714286,5.591837,5.101195,4.696520,4.860699,4.445996,4.272728,4.173890,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
584,NaN,200WEEW,5.054227,5.153390,4.398210,4.774973,4.965671,5.120336,4.776854,4.671114,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [8]:
# Select only needed columns
keep_columns = awc_keep_columns.append(pd.Index(["BWC_status_binary"]))
df_early_streaming_t201 = df_early_streaming_t201[keep_columns]
df_early_streaming_t201

,STUDENT_ID,GH 1,GH 2,GH 3,GH 4,GH 5,GH 6,GH 7,GH 8,GH 9,...,IF 1,IF 2,IF 3,IF 4,IF 5,IF 6,IFT,AWC_stream,AWC_stream_binary,BWC_status_binary
0,160CHANC,4.215324,3.937536,4.000000,3.736023,3.726859,3.747200,3.210893,2.673016,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,160CHANG,3.926446,3.746286,4.094832,4.036967,4.111887,3.783657,2.528977,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,160CHOOC,4.243429,4.379660,4.674136,4.110342,3.962646,3.928343,4.000000,3.901540,4.304875,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,1.0,1
3,160HOS,4.572844,4.476572,4.458939,3.850272,4.074621,4.110372,4.385034,4.229443,4.486146,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,1.0,1
4,160HOWW,4.936332,4.061821,3.903645,3.581086,3.133589,3.390447,1.487395,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581,200TERJ,4.858383,5.098765,4.722091,4.293651,4.182033,3.743067,4.467533,3.699077,4.191245,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
582,200TEWJ,5.813411,5.653140,5.094496,5.210907,4.576538,5.086124,4.853173,4.721885,4.454959,...,5.552131,5.111225,4.810760,4.898658,4.164966,4.266431,4.406520,5.0,1.0,1
583,200VVARUN,5.714286,5.591837,5.101195,4.696520,4.860699,4.445996,4.272728,4.173890,4.412128,...,4.850349,4.607360,4.696551,4.388513,4.277323,4.166171,4.347469,3.0,0.0,1
584,200WEEW,5.054227,5.153390,4.398210,4.774973,4.965671,5.120336,4.776854,4.671114,3.729477,...,4.672861,4.656508,4.441327,4.000000,3.945963,3.923590,4.511375,3.0,0.0,1


## Raw data for BWC 201 to 208 (source: LM)

In [9]:
df_201t208 = pd.read_csv("../data/LM_201t208data_upto2026March.csv") # read all data as strings to avoid type mismatch issues
df_201t208

/var/folders/bk/h4fkjwr93xs0lvghl5fm63l40000gn/T/ipykernel_26616/3283921897.py:1: DtypeWarning: Columns (0: COURSE_ACTUAL_END_DATE, 1: LESSON_TEMPLATE_COMMENTS, 2: ABORT_REASON, 3: ABORT_CODE, 4: EMPLOYEE_ACKNOWLEDGED_DATE, 5: EMP_ACK_BY_EMPLOYEE_DS_ID) have mixed types. Specify dtype option on import or set low_memory=False.
  df_201t208 = pd.read_csv("../data/LM_201t208data_upto2026March.csv") # read all data as strings to avoid type mismatch issues


,STUDENT_ID,STUDENT_FIRST_NAME,STUDENT_LAST_NAME,STUDENT_DISPLAY_NAME,STUDENT_TITLE,STUDENT_RANK,STUDENT_RANK_ABBREV,COURSE,CLASS_TEMP_DS_ID,CLASS,...,OBJECTIVE_DESCRIPTION,OBJECTIVE_RAW_SCORE,OBJECTIVE_WEIGHTED_SCORE,OBJECTIVE_SCORE_WEIGHT,OBJECTIVE_CRITICAL,OBJECTIVE_ACCEPTABLE_SCORE,OBJECTIVE_EXPECTED_SCORE,OBJECTIVE_ACCEPTABLE_SCORE_DSC,OBJECTIVE_EXPECTED_SCORE_DSC,OBJECTIVE_COMMENTS
0,201EEZ,Z,EE,EE Z,Trainee,NaN,NaN,NaN,NaN,201 BWC-B,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,201EEZ,Z,EE,EE Z,Trainee,NaN,NaN,NaN,NaN,201 BWC-B,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,201EEZ,Z,EE,EE Z,Trainee,NaN,NaN,NaN,NaN,201 BWC-B,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,201EEZ,Z,EE,EE Z,Trainee,NaN,NaN,NaN,NaN,201 BWC-B,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,201EEZ,Z,EE,EE Z,Trainee,NaN,NaN,NaN,NaN,201 BWC-B,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163186,208WONGZ,Z,WONG,WONG Z,Trainee,NaN,NaN,207 BWC-B,NaN,208 BWC-B,...,Arrival Procedures,3.0,NaN,NaN,False,NaN,3.0,NaN,NaN,Directed for Overhead 08
163187,208WONGZ,Z,WONG,WONG Z,Trainee,NaN,NaN,207 BWC-B,NaN,208 BWC-B,...,Climb,5.0,NaN,NaN,False,NaN,4.0,NaN,NaN,NaN
163188,208WONGZ,Z,WONG,WONG Z,Trainee,NaN,NaN,207 BWC-B,NaN,208 BWC-B,...,Descend,5.0,NaN,NaN,False,NaN,4.0,NaN,NaN,NaN
163189,208WONGZ,Z,WONG,WONG Z,Trainee,NaN,NaN,207 BWC-B,NaN,208 BWC-B,...,Straight and Level,5.0,NaN,NaN,False,NaN,4.0,NaN,NaN,NaN


### Cleaning and restructuring of data
NOTE: Code below adopted from `01_data.ipynb`

#### Drop duplicate rows

In [10]:
df_201t208_clean = df_201t208.drop_duplicates().reset_index(drop = True)
print(f"{len(df_201t208) - len(df_201t208_clean)} duplicate rows dropped. {len(df_201t208_clean)} rows remain.")

75 duplicate rows dropped. 163116 rows remain.


#### Drop rows with null values in `STUDENT_ID`, `EMP_EVAL_SCORE`, `LESSON_NUMBER`

In [11]:
bef = len(df_201t208_clean)
df_201t208_clean = df_201t208_clean.dropna(subset=["STUDENT_ID", "EMP_EVAL_SCORE", "LESSON_NUMBER"]).reset_index(drop = True)
aft = len(df_201t208_clean)
print(f"{bef - aft} rows with null values in STUDENT_ID, EMP_EVAL_SCORE, or LESSON_NUMBER dropped. {aft} rows remain.")

13196 rows with null values in STUDENT_ID, EMP_EVAL_SCORE, or LESSON_NUMBER dropped. 149920 rows remain.


#### Clean `CLASS` column

In [12]:
assert df_201t208_clean["CLASS"].isnull().sum() == 0

In [13]:
# Clean class column - replace weird values found by manual inspection
df_201t208_clean["CLASS"] = df_201t208_clean["CLASS"].str.replace("192B BWC-B", "192 BWC-B", regex=False)
df_201t208_clean["CLASS"] = df_201t208_clean["CLASS"].str.replace("202 BWC-B - A", "202 BWC-B", regex=False)

# Remove trailing whitespace
df_201t208_clean["CLASS"] = df_201t208_clean["CLASS"].str.strip()

#### Keep only rows belonging to BWS pilot trainees

In [14]:
# Value in CLASS must follow "XXX BWC-B" format
CLASS_OK_PATTERN = re.compile(r"^\d+\s+BWC-B$")

df_bwc = df_201t208_clean.loc[
    df_201t208_clean["CLASS"].astype("string").str.strip().str.match(CLASS_OK_PATTERN, na=False)
].reset_index(drop=True)

print(f"{len(df_201t208_clean) - len(df_bwc)} rows that are NOT from BWC pilot trainees dropped. These rows belong to BWC WSO trainees, or BWC-F trainees.")

57158 rows that are NOT from BWC pilot trainees dropped. These rows belong to BWC WSO trainees, or BWC-F trainees.


#### Creat `bwc_batch` column

In [15]:
# Extract numeric prefix at start of CLASS (e.g., "199" from "199 BWC-B")
df_bwc["bwc_batch"] = (
    df_bwc["CLASS"]
    .astype(str)
    .str.extract(r"^(\d+)", expand=False)  # capture leading digits only
    .astype("float")                       # convert to numeric type
    .astype("Int64")                       # optional: use pandas nullable int type
)

#### Clean `LESSON_NUMBER` column

In [16]:
def clean_lesson_numbers(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize LESSON_NUMBER strings.

    Rules:
    - drop rows where LESSON_NUMBER is NaN
    - remove leading batch prefixes like '150-' or '150- '
    - canonicalize common aliases
    - normalize suffix terms
    - insert spaces between letters and digits
    - collapse whitespace and trim
    """
    if "LESSON_NUMBER" not in df.columns:
        raise KeyError("Expected column 'LESSON_NUMBER' not found.")

    out_df = df.copy()
    out_df = out_df.dropna(subset=["LESSON_NUMBER"]).copy()

    s = out_df["LESSON_NUMBER"].astype("string").str.upper().str.strip()

    # Remove leading batch prefixes like:
    # 152-GH27R, 153- GH 1, 156 GH40FL
    s = s.str.replace(r"^\d+\s*-\s*", "", regex=True)
    s = s.str.replace(r"^\d+\s+", "", regex=True)

    # Remove leading MS prefixes like:
    # MS - GH 1, MS -GH 1, MS-GH 1
    s = s.str.replace(r"^MS\s*-\s*", "", regex=True)

    # Normalize known raw variants
    s = s.str.replace(r"\bG\s*/\s*S\b", "GS", regex=True)
    s = s.str.replace(r"\bFLEX\b", "FL", regex=True)

    # Canonicalize CHECK variants
    s = s.str.replace(r"\bCHECK\b", "CHK", regex=True)
    # GH 9 was originally PROG CHK
    s = s.str.replace(r"\bPROG\s+CHK\b", "GH 9", regex=True)

    # Canonicalize CO/OC check variants
    s = s.str.replace(
        r"^(OC|CO)(\s*/\s*(OC|CO))?\s+CHK",
        "CO/OC CHK",
        regex=True,
    )

    # Fix compact RV forms and known weird value
    s = s.str.replace(r"\bRV\s*1\s*-\s*19-05-2015\b", "RV 1", regex=True)
    s = s.str.replace(r"\bRV1\b", "RV 1", regex=True)
    s = s.str.replace(r"\bRV2\b", "RV 2", regex=True)
    s = s.str.replace(r"\bRV3\b", "RV 3", regex=True)

    # Insert spaces between letters and digits
    # GH27 -> GH 27, OFS10 -> OFS 10, GH27R -> GH 27 R
    s = s.str.replace(r"(?<=[A-Z])(?=\d)", " ", regex=True)
    s = s.str.replace(r"(?<=\d)(?=[A-Z])", " ", regex=True)

    # Canonicalize lesson aliases after spacing normalization
    # GH 35 was originally GHT
    # GH 40 was originally BHT
    s = s.str.replace(r"\bGH\s+35\b", "GHT", regex=True)
    s = s.str.replace(r"\bGH\s+39\b", "BHT", regex=True)
    s = s.str.replace(r"\bGH\s+40\b", "BHT", regex=True)

    # Normalize compact suffix forms
    s = s.str.replace(r"\bGHTR\b", "GHT R", regex=True)
    s = s.str.replace(r"\bIFTR\b", "IFT R", regex=True)

    # Normalize "FM FL 1" -> "FM 1 FL", "FM FL 2" -> "FM 2 FL"
    s = s.str.replace(r"\bFM\s+FL\s+1\b", "FM 1 FL", regex=True)
    s = s.str.replace(r"\bFM\s+FL\s+2\b", "FM 2 FL", regex=True)

    # Normalize "NG FL 1" -> "NG 1 FL", "NG FL 2" -> "NG 2 FL"
    s = s.str.replace(r"\bNG\s+FL\s+1\b", "NG 1 FL", regex=True)
    s = s.str.replace(r"\bNG\s+FL\s+2\b", "NG 2 FL", regex=True)

    # Normalize "GH FL 1" -> "GH 1 FL", "GH FL 2" -> "GH 2 FL"
    s = s.str.replace(r"\bGH\s+FL\s+1\b", "GH 1 FL", regex=True)
    s = s.str.replace(r"\bGH\s+FL\s+2\b", "GH 2 FL", regex=True)

    # Normalize whitespace
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()

    out_df["LESSON_NUMBER"] = s
    return out_df

df_201t208_clean = clean_lesson_numbers(df_bwc)

#### Create `lesson_suffix` and `lesson_base` columns

In [17]:
def add_lesson_suffix(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add lesson_suffix from LESSON_NUMBER.

    lesson_suffix is:
      - 'R', 'FL', 'GS', or 'CX', 'OC' if the final token is one of these
      - <NA> otherwise
    """
    if "LESSON_NUMBER" not in df.columns:
        raise KeyError("Expected column 'LESSON_NUMBER' not found.")

    out_df = df.copy()

    s = out_df["LESSON_NUMBER"].astype("string").str.upper().str.strip()

    out_df["lesson_suffix"] = s.str.extract(r"\b(R|FL|GS|CX|OC)$", expand=False).astype(
        "string"
    )

    return out_df


def add_lesson_base(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add lesson_base from LESSON_NUMBER by removing final suffix token
    if it is one of R, FL, GS, CX, OC.
    """
    if "LESSON_NUMBER" not in df.columns:
        raise KeyError("Expected column 'LESSON_NUMBER' not found.")

    out_df = df.copy()

    s = out_df["LESSON_NUMBER"].astype("string").str.upper().str.strip()
    
    out_df["lesson_base"] = s.str.replace(
        r"(?:\s+(?:R|FL|GS|CX|OC))+$",
        "",
        regex=True,
    )

    return out_df

df_201t208_clean = add_lesson_suffix(df_201t208_clean)
df_201t208_clean = add_lesson_base(df_201t208_clean)

#### Drop rows with errors indicated by `EMP_EVAL_COMMENTS`

In [18]:
bef = len(df_201t208_clean)
df_201t208_clean = df_201t208_clean[~df_201t208_clean["EMP_EVAL_COMMENTS"].str.contains("Duplicated event", case=False, regex=False, na=False)]
df_201t208_clean = df_201t208_clean[~df_201t208_clean["EMP_EVAL_COMMENTS"].str.contains("Double up gradesheet", case=False, regex=False, na=False)]
df_201t208_clean = df_201t208_clean[~df_201t208_clean["EMP_EVAL_COMMENTS"].str.contains("Erroneous gradesheet", case=False, regex=False, na=False)]
df_201t208_clean = df_201t208_clean[~df_201t208_clean["EMP_EVAL_COMMENTS"].str.contains("error gradesheet", case=False, regex=False, na=False)]
df_201t208_clean = df_201t208_clean[~df_201t208_clean["EMP_EVAL_COMMENTS"].str.contains("error gs", case=False, regex=False, na=False)]
df_201t208_clean = df_201t208_clean[~df_201t208_clean["EMP_EVAL_COMMENTS"].str.contains("wrong gs", case=False, regex=False, na=False)]
df_201t208_clean = df_201t208_clean[df_201t208_clean["EMP_EVAL_COMMENTS"] != "ERROR"]
df_201t208_clean = df_201t208_clean[df_201t208_clean["EMP_EVAL_COMMENTS"] != "error"]
aft = len(df_201t208_clean)
print(f"{bef - aft} rows with errors indicated by EMP_EVAL_COMMENTS dropped. {aft} rows remain.")

78 rows with errors indicated by EMP_EVAL_COMMENTS dropped. 92684 rows remain.


#### Keep only relevant lesson numbers

In [19]:
bef = len(df_201t208_clean)
df_201t208_clean= df_201t208_clean[df_201t208_clean["lesson_base"].isin(LESSONS_TO_KEEP)].reset_index(drop=True)
aft = len(df_201t208_clean)
print(f"{bef - aft} rows belonging to lesson numbers NOT in modules 1 to 3 dropped. {aft} rows remain.")

54907 rows belonging to lesson numbers NOT in modules 1 to 3 dropped. 37777 rows remain.


#### Drop rows with `lesson_suffix` "R", "GS", "FL"

In [20]:
bef = len(df_201t208_clean)
df_201t208_clean = df_201t208_clean[~df_201t208_clean["lesson_suffix"].isin(["R", "GS", "FL"])].reset_index(drop=True)
aft = len(df_201t208_clean)
print(f"{bef - aft} rows with LESSON_NUMBER suffixed with R, GS, FL dropped. {aft} rows remain.")

820 rows with LESSON_NUMBER suffixed with R, GS, FL dropped. 36957 rows remain.


#### Drop rows with "NG"
If `EMP_EVAL_COMMENTS` contain "ng" or "NG" \
**OR** `lesson_suffix` is "GS", the sortie is not graded.

In [21]:
# Create helper ng boolean column
df_201t208_clean["ng"] = (
    df_201t208_clean["EMP_EVAL_COMMENTS"]
    .astype("string")
    .str.contains(r"\bNG\b", case=False, na=False)
)

# Check for ghosted solo in the comments, as some LESSON_NUMBERs have indicators of
# ghosted solo in the comments but have not been suffixed by GS.
mask = df_201t208_clean["EMP_EVAL_COMMENTS"].astype("string").str.contains(r"Ghost", case=False, na=False)
df_201t208_clean.loc[mask, "lesson_suffix"] = "GS"
needs_gs = mask & ~df_201t208_clean["LESSON_NUMBER"].astype("string").str.endswith(" GS", na=False)
df_201t208_clean.loc[needs_gs, "LESSON_NUMBER"] = df_201t208_clean.loc[needs_gs, "LESSON_NUMBER"].astype("string") + " GS"

# Correct after manual inspection using excel
mask = (df_201t208_clean["STUDENT_ID"] == "181GOHR") & (df_201t208_clean["LESSON_NUMBER"] == "GH 22 GS")
df_201t208_clean.loc[mask, "LESSON_NUMBER"] = "GH 22"
df_201t208_clean.loc[mask, "lesson_suffix"] = pd.NA
mask = (df_201t208_clean["STUDENT_ID"] == "199LIMH") & (df_201t208_clean["LESSON_NUMBER"] == "GH 22 GS")
df_201t208_clean.loc[mask, "LESSON_NUMBER"] = "GH 22"
df_201t208_clean.loc[mask, "lesson_suffix"] = pd.NA
mask = (df_201t208_clean["STUDENT_ID"] == "202WONGS") & (df_201t208_clean["LESSON_NUMBER"] == "GH 25 GS")
df_201t208_clean.loc[mask, "LESSON_NUMBER"] = "GH 25"
df_201t208_clean.loc[mask, "lesson_suffix"] = pd.NA

# Ghosted solos are not graded.
mask = df_201t208_clean["lesson_suffix"] == "GS"
df_201t208_clean.loc[mask, "ng"] = True

# Drop rows with NG
bef = len(df_201t208_clean)
df_201t208_clean = df_201t208_clean[~df_201t208_clean["ng"]].reset_index(drop=True)
aft = len(df_201t208_clean)
print(f"{bef - aft} rows with NG indicators have been dropped. {aft} rows remain.")

# Remove helper column
df_201t208_clean = df_201t208_clean.drop(columns=["ng"])

3108 rows with NG indicators have been dropped. 33849 rows remain.


#### Drop rows with "DNCO"

In [22]:
# Create boolean helper column
# Condition: If DUTY_STATUS_DESCRIPTION == DNCO, or if EMP_EVAL_COMMENTS contain DNCO
df_201t208_clean["dnco"] = (
    df_201t208_clean["DUTY_STATUS_DESCRIPTION"].astype("string").eq("DNCO")
    |
    df_201t208_clean["EMP_EVAL_COMMENTS"]
    .astype("string")
    .str.contains(r"\bDNCO\b", case=False, na=False)
)
df_201t208_clean["dnco"] = df_201t208_clean["dnco"].fillna(False)

# Drop rows with DNCO
bef = len(df_201t208_clean)
df_201t208_clean = df_201t208_clean[~df_201t208_clean["dnco"]].reset_index(drop=True)
aft = len(df_201t208_clean)
print(f"{bef - aft} rows with DNCO indicators have been dropped. {aft} rows remain.")

# Remove helper column
df_201t208_clean = df_201t208_clean.drop(columns=["dnco"])

84 rows with DNCO indicators have been dropped. 33765 rows remain.


#### Keep only the first row of each LESSON_NUMBER for each STUDENT_ID
The data contains one row for each objective of a LESSON_NUMBER. All rows belonging to the same LESSON_NUMBER should have the same EMP_EVAL_SCORE. If duplicate EMP_EVAL_SCORE values exist, keep the first instance.

In [23]:
df_filtered = df_201t208_clean.drop_duplicates(subset=["STUDENT_ID", "LESSON_NUMBER", "LESSON_ACTUAL_START_DATE"]).reset_index(drop=True)

#### Drop duplicate combinations of `STUDENT_ID`, `LESSON_NUMBER`

At this point, each `STUDENT_ID` should only have ONE row per `LESSON_NUMBER`.

In [24]:
group_sizes = df_filtered.groupby(["STUDENT_ID", "LESSON_NUMBER"]).size()
bad_groups = group_sizes[group_sizes > 1]
assert bad_groups.empty, f"Found duplicate groups:\n{bad_groups}"

AssertionError: Found duplicate groups:
STUDENT_ID  LESSON_NUMBER
201HONT     GH 15            2
201LEEJ     GH 15            2
            GH 25            2
202PHUAC    GH 15            2
202POHK     GH 5             2
208BHAVESH  GH 15            2
            GH 22            2
208JKOK     GH 4             2
208KOHZ     GH 5             2
208LAIL     GH 2             2
208NGOOIJ   GH 4             2
dtype: int64

In [25]:
bad_groups.sort_values(ascending=False).head(20)

STUDENT_ID  LESSON_NUMBER
201HONT     GH 15            2
201LEEJ     GH 15            2
            GH 25            2
202PHUAC    GH 15            2
202POHK     GH 5             2
208BHAVESH  GH 15            2
            GH 22            2
208JKOK     GH 4             2
208KOHZ     GH 5             2
208LAIL     GH 2             2
208NGOOIJ   GH 4             2
dtype: int64

In [26]:
mask = (df_filtered["STUDENT_ID"] == "168SOHJ") & (df_filtered["LESSON_NUMBER"] == "GH 17")
print(df_filtered[mask]["LESSON_ACTUAL_START_DATE"].unique().tolist())
print(df_filtered[mask]["EMP_EVAL_COMMENTS"].unique().tolist())
print(df_filtered[mask]["EMP_EVAL_SCORE"].unique().tolist())

[]
[]
[]


In [27]:
# Drop rows
bef = len(df_filtered)
df_filtered = df_filtered.drop_duplicates(subset=["STUDENT_ID", "LESSON_NUMBER"])
aft = len(df_filtered)
print(f"{bef - aft} rows dropped")

11 rows dropped


In [28]:
group_sizes = df_filtered.groupby(["STUDENT_ID", "LESSON_NUMBER"]).size()
bad_groups = group_sizes[group_sizes > 1]
assert bad_groups.empty, f"Found duplicate groups:\n{bad_groups}"

#### Pivot

In [29]:
df_201t208_clean = df_filtered.copy()

# Fillna in ADJUSTED_SCORE with EMP_EVAL_SCORE 
df_201t208_clean["ADJUSTED_SCORE"] = df_201t208_clean["ADJUSTED_SCORE"].fillna(df_201t208_clean["EMP_EVAL_SCORE"])

# Sanity check: Ensure that there is only one row per STUDENT_ID and LESSON_NUMBER
group_sizes = df_201t208_clean.groupby(["STUDENT_ID", "LESSON_NUMBER"]).size()
bad_groups = group_sizes[group_sizes > 1]
assert bad_groups.empty, f"Found duplicate groups:\n{bad_groups}"
assert df_201t208_clean["EMP_EVAL_SCORE"].isnull().sum() == 0
assert df_201t208_clean["ADJUSTED_SCORE"].isnull().sum() == 0

In [30]:
# Pivot table
wide = (
    df_201t208_clean.pivot_table(
        index="STUDENT_ID",
        columns="LESSON_NUMBER",
        values=["EMP_EVAL_SCORE", "ADJUSTED_SCORE"],
        aggfunc="first",
    )
)

# Reorder columns so lesson order is preserved, and within each lesson: score, adjscore
ordered_cols = [
    (value_col, lesson)
    for lesson in LESSONS_TO_KEEP
    for value_col in ["EMP_EVAL_SCORE", "ADJUSTED_SCORE"]
    if (value_col, lesson) in wide.columns
]
wide = wide.reindex(columns=ordered_cols)

# Flatten multiindex columns
name_map = {
    "EMP_EVAL_SCORE": "score",
    "ADJUSTED_SCORE": "adjscore",
}
wide.columns = [
    f"{lesson}_{name_map[value_col]}"
    for value_col, lesson in wide.columns
]
wide = wide.reset_index()
wide.info()

<class 'pandas.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 71 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   STUDENT_ID      110 non-null    str    
 1   GH 1_score      57 non-null     float64
 2   GH 1_adjscore   57 non-null     float64
 3   GH 2_score      63 non-null     float64
 4   GH 2_adjscore   63 non-null     float64
 5   GH 3_score      60 non-null     float64
 6   GH 3_adjscore   60 non-null     float64
 7   GH 4_score      59 non-null     float64
 8   GH 4_adjscore   59 non-null     float64
 9   GH 5_score      52 non-null     float64
 10  GH 5_adjscore   52 non-null     float64
 11  GH 6_score      49 non-null     float64
 12  GH 6_adjscore   49 non-null     float64
 13  GH 7_score      52 non-null     float64
 14  GH 7_adjscore   52 non-null     float64
 15  GH 8_score      46 non-null     float64
 16  GH 8_adjscore   46 non-null     float64
 17  GH 9_score      61 non-null     float64
 18  G

In [31]:
# Select only _adjscore columns + STUDENT_ID
adj_cols = ['STUDENT_ID'] + [col for col in wide.columns if col.endswith('_adjscore')]
wide = wide[adj_cols]

# Drop GHT
wide = wide.drop(columns=['GHT_adjscore'])

# Rename: strip '_adjscore' suffix
rename_map = {col: col.replace('_adjscore', '') for col in wide.columns if col.endswith('_adjscore')}
wide = wide.rename(columns=rename_map)

#### Append ground truth labels

In [32]:
gt_201t208_df = pd.read_excel("../data/BWC201t208_Status_as_of_end_Mar2026.xlsx")

AWC_stream	av_creoc	AWC_stream_binary

In [33]:
gt_201t208_df["BWC_Status"] = gt_201t208_df["BWC_Status"].map(BWC_status_dict)
gt_201t208_df = gt_201t208_df.rename(columns={"STUDENT_ID_ORIG": "STUDENT_ID"})

In [34]:
df_201t208_clean = wide.merge(
    gt_201t208_df,
    on = "STUDENT_ID",
    how = "left",
)
df_201t208_clean

,STUDENT_ID,GH 1,GH 2,GH 3,GH 4,GH 5,GH 6,GH 7,GH 8,GH 9,...,IF 2,IF 3,IF 4,IF 5,IF 6,IFT,Batch,BWC_Status,Stream_Group,Stream_Unit
0,201EEZ,NaN,NaN,5.234087,NaN,NaN,4.367075,NaN,NaN,4.135952,...,NaN,NaN,NaN,NaN,NaN,NaN,201,0.0,NaN,NaN
1,201HLEE,4.624989,NaN,NaN,NaN,NaN,NaN,4.155844,4.118668,5.054041,...,NaN,NaN,NaN,NaN,NaN,NaN,201,0.0,NaN,NaN
2,201HONT,5.113508,NaN,NaN,NaN,NaN,4.735986,NaN,NaN,4.000000,...,4.245510,4.539404,4.406282,3.777039,NaN,NaN,201,0.0,NaN,NaN
3,201KHOOC,NaN,NaN,NaN,4.634096,NaN,NaN,NaN,NaN,4.272362,...,5.277256,NaN,4.386612,NaN,NaN,4.204591,201,1.0,Heli,RWC
4,201KOHZ,NaN,NaN,5.756446,4.930210,4.751955,NaN,NaN,NaN,5.286757,...,4.491836,4.585104,3.000000,NaN,3.785039,NaN,201,1.0,Fighter,FWC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,208TEOC,5.610000,5.84,4.710000,5.210000,4.900000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,208,NaN,NaN,NaN
106,208TOHZ,5.230000,5.13,3.000000,3.000000,3.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,208,NaN,NaN,NaN
107,208TONGZ,5.900000,5.99,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,208,NaN,NaN,NaN
108,208WONGW,5.520000,5.47,4.800000,3.000000,4.410000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,208,NaN,NaN,NaN


In [35]:
df_201t208_clean = df_201t208_clean.rename(columns={"BWC_Status": "BWC_status_binary",
                                                    "Stream_Group": "av_creoc",
                                                    "Stream_Unit": "AWC_stream"})
df_201t208_clean

,STUDENT_ID,GH 1,GH 2,GH 3,GH 4,GH 5,GH 6,GH 7,GH 8,GH 9,...,IF 2,IF 3,IF 4,IF 5,IF 6,IFT,Batch,BWC_status_binary,av_creoc,AWC_stream
0,201EEZ,NaN,NaN,5.234087,NaN,NaN,4.367075,NaN,NaN,4.135952,...,NaN,NaN,NaN,NaN,NaN,NaN,201,0.0,NaN,NaN
1,201HLEE,4.624989,NaN,NaN,NaN,NaN,NaN,4.155844,4.118668,5.054041,...,NaN,NaN,NaN,NaN,NaN,NaN,201,0.0,NaN,NaN
2,201HONT,5.113508,NaN,NaN,NaN,NaN,4.735986,NaN,NaN,4.000000,...,4.245510,4.539404,4.406282,3.777039,NaN,NaN,201,0.0,NaN,NaN
3,201KHOOC,NaN,NaN,NaN,4.634096,NaN,NaN,NaN,NaN,4.272362,...,5.277256,NaN,4.386612,NaN,NaN,4.204591,201,1.0,Heli,RWC
4,201KOHZ,NaN,NaN,5.756446,4.930210,4.751955,NaN,NaN,NaN,5.286757,...,4.491836,4.585104,3.000000,NaN,3.785039,NaN,201,1.0,Fighter,FWC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,208TEOC,5.610000,5.84,4.710000,5.210000,4.900000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,208,NaN,NaN,NaN
106,208TOHZ,5.230000,5.13,3.000000,3.000000,3.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,208,NaN,NaN,NaN
107,208TONGZ,5.900000,5.99,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,208,NaN,NaN,NaN
108,208WONGW,5.520000,5.47,4.800000,3.000000,4.410000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,208,NaN,NaN,NaN


### Remapping of `AW_stream` to `AWC_stream_binary`

In [36]:
df_201t208_clean["AWC_stream"] = df_201t208_clean["AWC_stream"].map(stream_unit_dict)
df_201t208_clean["AWC_stream_binary"] = df_201t208_clean["AWC_stream"].map(AWC_stream_binary_dict)
df_201t208_clean

,STUDENT_ID,GH 1,GH 2,GH 3,GH 4,GH 5,GH 6,GH 7,GH 8,GH 9,...,IF 3,IF 4,IF 5,IF 6,IFT,Batch,BWC_status_binary,av_creoc,AWC_stream,AWC_stream_binary
0,201EEZ,NaN,NaN,5.234087,NaN,NaN,4.367075,NaN,NaN,4.135952,...,NaN,NaN,NaN,NaN,NaN,201,0.0,NaN,NaN,NaN
1,201HLEE,4.624989,NaN,NaN,NaN,NaN,NaN,4.155844,4.118668,5.054041,...,NaN,NaN,NaN,NaN,NaN,201,0.0,NaN,NaN,NaN
2,201HONT,5.113508,NaN,NaN,NaN,NaN,4.735986,NaN,NaN,4.000000,...,4.539404,4.406282,3.777039,NaN,NaN,201,0.0,NaN,NaN,NaN
3,201KHOOC,NaN,NaN,NaN,4.634096,NaN,NaN,NaN,NaN,4.272362,...,NaN,4.386612,NaN,NaN,4.204591,201,1.0,Heli,2.0,0.0
4,201KOHZ,NaN,NaN,5.756446,4.930210,4.751955,NaN,NaN,NaN,5.286757,...,4.585104,3.000000,NaN,3.785039,NaN,201,1.0,Fighter,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,208TEOC,5.610000,5.84,4.710000,5.210000,4.900000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,208,NaN,NaN,NaN,NaN
106,208TOHZ,5.230000,5.13,3.000000,3.000000,3.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,208,NaN,NaN,NaN,NaN
107,208TONGZ,5.900000,5.99,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,208,NaN,NaN,NaN,NaN
108,208WONGW,5.520000,5.47,4.800000,3.000000,4.410000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,208,NaN,NaN,NaN,NaN


### Only keep columns in `keep_columns` (extracted from Cleaned data for BWC 106 to 200)

In [37]:
df_201t208_clean = df_201t208_clean.reindex(columns = keep_columns)
df_201t208_clean.columns

Index(['STUDENT_ID', 'GH 1', 'GH 2', 'GH 3', 'GH 4', 'GH 5', 'GH 6', 'GH 7',
       'GH 8', 'GH 9', 'GH 10', 'GH 11', 'GH 12', 'GH 13', 'GH 14', 'GH 15',
       'GH 17', 'GH 19', 'GH 21', 'GH 22', 'GH 24', 'GH 25', 'GH 27', 'GH 28',
       'GH 29', 'GH 30', 'GH 31', 'GH 32', 'GH 33', 'GH 34', 'GH 35', 'IF 1',
       'IF 2', 'IF 3', 'IF 4', 'IF 5', 'IF 6', 'IFT', 'AWC_stream',
       'AWC_stream_binary', 'BWC_status_binary'],
      dtype='str')

## Writing of DataFrames into csv

In [38]:

df_early_streaming_t201.to_csv("../data/cleaned_early_streaming_t201.csv", index=False)
df_201t208_clean.to_csv("../data/cleaned_201t208.csv", index=False)
